##### RAG 시스템 비교 분석: Baseline vs Improved (Multi-Query)

이 노트북은 RAG(Retrieval-Augmented Generation) 시스템의 성능을 평가하고 개선하는 과정을 담고 있습니다. 
Ragas 라이브러리를 사용하여 성능 지표를 측정하며, Baseline 시스템과 Multi-Query가 적용된 개선 시스템을 비교합니다.

In [42]:
from quest_rag import *

# nest_asyncio.apply()
# load_dotenv()

In [44]:
file_path = "data/dog_wiki.txt" 

# 고난도 데이터셋 설정
eval_questions = [
    "강아지의 유치 개수와 성견의 영구치 개수 차이를 구하고, 어금니가 돋아나는 시점을 기준으로 한 성장 단계를 설명하시오.",
    "개의 기원과 관련하여 회색늑대와의 유전적 관계 및 가축화 시기에 대한 학계의 통설을 기술하시오.",
    "강아지의 감각 기능 중 청각과 후각의 특징을 인간과 비교하여 기술하고, 특히 가청 주파수의 범위를 수치로 제시하시오.",
    "문서에 언급된 '강아지'라는 단어의 어원적 구성 요소를 분석하고, 현대 국어에서 '망아지'와의 공통적인 형성 원리를 설명하시오.",
    "강아지의 지능 발달 수준을 인간의 연령과 비교하고, 지능 수준에 영향을 미치는 요인 3가지를 본문 내용에 근거해 제시하시오.",
    "강아지의 성장 과정 중 '사회화 시기'가 갖는 중요성과 이 시기에 적절한 자극을 받지 못했을 때 발생할 수 있는 부작용을 서술하시오.",
    "강아지의 신체 부위 중 발바닥(Pad)의 해부학적 기능과 땀샘의 위치가 체온 조절에 미치는 영향을 설명하시오.",
    "개의 수명에 영향을 미치는 체구(크기)별 상관관계를 설명하고, 소형견과 대형견 중 어느 쪽이 일반적으로 더 장수하는지 기술하시오.",
    "강아지의 영양 섭취에 있어 반드시 피해야 할 음식물 중 '초콜릿'과 '양파'가 신체에 미치는 치명적인 영향의 차이를 기술하시오.",
    "본문에서 '강아지'라는 명칭이 현대 사회에서 성견까지 포함한 반려견 전체를 지칭하는 용어로 확장된 국어학적 배경을 설명하시오."
]

ground_truths = [
    "강아지의 유치는 28개이며, 성견의 영구치는 42개로 총 14개의 차이가 난다. 생후 4개월 무렵부터 유치가 빠지기 시작하며 영구치인 어금니가 돋아나기 시작한다.",
    "개는 약 1만 5천 년 전 또는 그 이전부터 회색늑대를 조상으로 하여 가축화되었다. 이는 인류가 농경 생활을 시작하기 전인 수렵 채집 단계에서 이루어진 것으로 보고 있다.",
    "후각 세포는 약 2억 개로 인간보다 40배 이상 발달했다. 청각의 가청 주파수는 약 15~30,000Hz(최대 45,000Hz 이상)로 인간(20~20,000Hz)보다 훨씬 넓은 범위를 인지한다.",
    "'강아지'는 '개'에 짐승의 새끼를 뜻하는 접미사 '-아지'가 붙은 형태이다. 이는 '말'에 '-아지'가 붙은 '망아지'와 동일한 단어 형성 원리를 따른다.",
    "강아지의 지능은 인간의 2~3세 유아와 비슷하다. 지능에 영향을 미치는 주요 요인으로는 품종(유전적 요인), 성장 환경, 그리고 사회화 교육이 있다.",
    "생후 3~12주 사이의 사회화 시기에 외부 자극이나 타인과의 교감이 부족할 경우, 성견이 되었을 때 낯선 대상에 대해 극심한 공포심이나 공격적 성향을 보일 수 있다.",
    "발바닥 패드는 지면의 충격을 흡수하고 발을 보호한다. 개는 땀샘이 발바닥에만 집중되어 있어 땀을 통한 체온 조절 능력이 낮으며, 이를 보완하기 위해 입으로 헐떡이며 열을 배출한다.",
    "일반적으로 소형견이 대형견보다 수명이 길다. 대형견은 신체 대사율이 높고 세포 분열 속도가 빨라 노화가 소형견보다 일찍 진행되는 경향이 있기 때문이다.",
    "초콜릿의 테오브로민 성분은 신경계 및 심장에 중독을 일으키며, 양파나 마늘에 포함된 성분은 강아지의 적혈구를 파괴하여 용혈성 빈혈을 유발한다.",
    "본래 개의 새끼를 뜻하는 단어였으나, 대상에 대한 친근함과 애정을 담아 부르는 애칭으로 쓰이면서 현대에는 성견을 포함한 반려견 전체를 일컫는 완곡한 표현으로 의미가 확장되었다."
]

In [45]:
cfg_rag = {
    "title": "RAG Config",
    "model_name": "gpt-4o-mini",
    "embedding_model_name": "text-embedding-3-small",
    "chunk_size": 1000,
    "chunk_overlap": 100
}
print_cfg(cfg_rag)
rag = RAGProcessor(file_path, cfg_rag)
evaluator = RAGEvaluator(llm=rag.llm, embeddings=rag.embedding_model)


 [Config: RAG Config] 
 - model_name: gpt-4o-mini
 - embedding_model_name: text-embedding-3-small
 - chunk_size: 1000
 - chunk_overlap: 100
--------------------------------------------------


##### 5. Baseline 시스템 평가

In [ ]:
# 1. Baseline 파이프라인 실행 및 평가
cfg = {
    "title": "Baseline", 
    "k": 3, 
    "use_multi_query": False,
    "search_type": "similarity",
    "search_kwargs": {"k": 3}
}
chain, retriever = rag.get_rag_chain(cfg)
res = run_rag_pipeline(evaluator, chain, retriever, eval_questions, ground_truths, cfg)

df = pd.DataFrame([res])
print(df)


 [Config: Baseline] 
 - k: 3
 - use_multi_query: False
 - search_type: similarity
 - search_kwargs: {'k': 3}
--------------------------------------------------
--- Evaluating Baseline ---


Evaluating:   2%|▎         | 1/40 [00:02<01:53,  2.90s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 40/40 [01:13<00:00,  1.84s/it]


   faithfulness  answer_relevancy  context_recall  context_precision
0      0.484167          0.294777            0.25           0.333333


##### 6. 추가 실험 (Multi-Query, MMR 등)

In [47]:
# 2. MMR 실험 실행 및 평가
cfg = {
    "title": "Improved + MMR", 
    "k": 3, 
    "use_multi_query": True, 
    "search_type": "mmr", 
    "search_kwargs": {"k": 3, "fetch_k": 10}
}
chain, retriever = rag.get_rag_chain(cfg)
res = run_rag_pipeline(evaluator, chain, retriever, eval_questions, ground_truths, cfg)

df = pd.DataFrame([res])
print(df)


 [Config: Improved + MMR] 
 - k: 3
 - use_multi_query: True
 - search_type: mmr
 - search_kwargs: {'k': 3, 'fetch_k': 10}
--------------------------------------------------
--- Evaluating Improved + MMR ---


Evaluating:   2%|▎         | 1/40 [00:02<01:52,  2.88s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 40/40 [01:05<00:00,  1.63s/it]


   faithfulness  answer_relevancy  context_recall  context_precision
0      0.588052          0.366436             0.3                0.4


In [48]:
cfg = {
    "title": "Improved + MMR", 
    "k": 3, 
    "use_multi_query": True, 
    "search_type": "mmr", 
    "search_kwargs": {"k": 3, "fetch_k": 10}
}
chain, retriever = rag.get_rag_chain(cfg)
res = run_rag_pipeline(evaluator, chain, retriever, eval_questions, ground_truths, cfg)

df = pd.DataFrame([res])
print(df)


 [Config: Improved + MMR] 
 - k: 3
 - use_multi_query: True
 - search_type: mmr
 - search_kwargs: {'k': 3, 'fetch_k': 10}
--------------------------------------------------
--- Evaluating Improved + MMR ---


Evaluating:   2%|▎         | 1/40 [00:02<01:48,  2.78s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 40/40 [01:09<00:00,  1.73s/it]


   faithfulness  answer_relevancy  context_recall  context_precision
0      0.598929          0.357741             0.3           0.416667
